<a href="https://colab.research.google.com/github/krishnamurtv1/Codepath/blob/main/CSIT553_class3_2_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSIT553 Exercise: Aggregation & Pivot Table

In [1]:
# only run once
!git clone https://github.com/profliuhao/CSIT553.git

Cloning into 'CSIT553'...
remote: Enumerating objects: 190, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 190 (delta 16), reused 2 (delta 2), pack-reused 160 (from 1)
Receiving objects: 100% (190/190), 5.74 MiB | 10.30 MiB/s, done.
Resolving deltas: 100% (83/83), done.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rc('figure', figsize=(10, 6))
PREVIOUS_MAX_ROWS = pd.options.display.max_rows
pd.options.display.max_rows = 20


In [3]:
# titanic = pd.read_csv('datasets/titanic/train.csv')

titanic = pd.read_csv('/content/CSIT553/Module_3/CSIT553_class3_2_exercise/train.csv')

In [4]:
titanic.shape

(891, 12)

In [5]:
titanic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


Survived: 1 = Yes, 0= No

Pclass (Passenger Class): 1,2,3

Sex: Male, Female

Embarked (Port of Embarkation): C = Cherbourg, Q = Queenstown, S = Southampton

In [6]:
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Exercise 1. Get the survival rate by gender

Create a Series with the survival rate of female and male.

In [8]:
survival_rate_by_gender = titanic.groupby('Sex')['Survived'].mean()
display(survival_rate_by_gender)

,Survived
Sex,
female,0.742038
male,0.188908


### Exercise 2. Get the survival rate by gender and class

Use **groupby** and **unstack** to create a data frame with the survival rate by gender (rows) and class (columns).

Pclass,1,2,3
Sex,,,
female,0.968085,0.921053,0.500000
male,0.368852,0.157407,0.135447


### Exercise 3. Get the survival rate by gender and class

Use **pivot_table** to create a data frame with the survival rate by gender (rows) and class (columns), and enable the margins.

Pclass,1,2,3,All
Sex,,,,
female,0.968085,0.921053,0.500000,0.742038
male,0.368852,0.157407,0.135447,0.188908
All,0.629630,0.472826,0.242363,0.383838


### Exercise 4. Get the survival rate by gender, class, and age
Use **pivot_table** to create a data frame of the survival rate with hierarchical indexing, gender and type(child or adult), and columns for classes.

In [18]:
def convert_age (row):
    if row['Age'] < 18:
        return 'Child'
    if row['Age'] >= 18:
        return 'Adult'
    else:
        return 'NA'

In [19]:
titanic['Type'] = titanic.apply(lambda row: convert_age (row),axis=1)

In [20]:
titanic = titanic[titanic["Type"] != 'NA']

In [21]:
#age = pd.cut(titanic['Age'], [0, 18, 80])

In [23]:
def convert_age (row):
    if row['Age'] < 18:
        return 'Child'
    if row['Age'] >= 18:
        return 'Adult'
    else:
        return 'NA'

if 'Type' not in titanic.columns:
    titanic['Type'] = titanic.apply(lambda row: convert_age(row), axis=1)

# Filter out 'NA' types if not already done, ensuring the Type column is clean for analysis.
# Create a copy to avoid SettingWithCopyWarning if original `titanic` DataFrame was a slice
titanic_filtered_by_type = titanic[titanic['Type'] != 'NA'].copy()

survival_table = pd.pivot_table(
    titanic_filtered_by_type,
    values='Survived',
    index=['Sex', 'Type'],
    columns='Pclass',
    aggfunc='mean'
)

display(survival_table)

Pclass               1         2         3
Sex    Type                               
female Adult  0.974026  0.903226  0.417910
       Child  0.875000  1.000000  0.542857
male   Adult  0.371134  0.068182  0.133333
       Child  1.000000  0.818182  0.232558

## Exercise 5

Sales Analysis with Pivot_table

Real-world scenario: You're a data analyst at a retail company that sells electronics across different regions. You have daily sales data for 2023-2024 that needs to be analyzed for the quarterly business review.

Task Description:

Using the sales dataset, create:

1. A quarterly sales table showing total sales by region
2. quarter-over-quarter growth analysis for each region
3. Identify the best and worst performing quarter in each region

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

days=366


# Creating realistic sales data
sales_data = pd.DataFrame({
    'Date': pd.date_range(start='2024-01-01', end='2024-12-31', freq='D'),
    'Region': np.random.choice(['North', 'South', 'East', 'West'], days),
    'Category': np.random.choice(['Laptops', 'Smartphones', 'Tablets', 'Accessories'], days),
    'Sales': np.random.randint(50000, 200000, days),
    'Units': np.random.randint(10, 1000, days)
})

print("Original Sales Data:")
print(sales_data)

# Task: Create a quarterly sales summary by region
sales_data['Quarter'] = pd.PeriodIndex(sales_data['Date'], freq='Q')
sales_data.head()


Original Sales Data:
          Date Region     Category   Sales  Units
0   2024-01-01   West      Laptops  193429    177
1   2024-01-02   East      Laptops  162712    781
2   2024-01-03  South      Tablets  162361    407
3   2024-01-04  North      Tablets   60664    286
4   2024-01-05   East  Smartphones  166749    284
..         ...    ...          ...     ...    ...
361 2024-12-27  South      Tablets   75695    711
362 2024-12-28   East      Tablets  102531    489
363 2024-12-29  North  Smartphones   81917     19
364 2024-12-30  North      Laptops   62129    329
365 2024-12-31  North      Laptops  196621    409

[366 rows x 5 columns]


,Date,Region,Category,Sales,Units,Quarter
0,2024-01-01,West,Laptops,193429,177,2024Q1
1,2024-01-02,East,Laptops,162712,781,2024Q1
2,2024-01-03,South,Tablets,162361,407,2024Q1
3,2024-01-04,North,Tablets,60664,286,2024Q1
4,2024-01-05,East,Smartphones,166749,284,2024Q1


In [ ]:
# your solution below:
# Step 2: Create pivot table
# Hint: Use index=?
# Hint: Use values=?




# Step 3: Calculate growth
# Hint: Use pct_change() method on the appropriate axis





## Exercise 6

Medical Research Data with Melt and Groupby

Patient Vital Signs Analysis

Medical Context: A clinical research team needs to analyze patterns in patient vital signs across different times of day and treatment groups.

Task Description: Transform the wide-format patient data to:

1. Create a long-format dataset suitable for time-series analysis
2. Enable comparison of vital signs across different times of day
3. Analyze the effect of treatment groups on vital signs

In [ ]:
# Creating a large patient monitoring dataset
np.random.seed(42)
num_patients = 500
days = 30

# Generate patient demographic data
patient_base = pd.DataFrame({
    'Patient_ID': range(1, num_patients + 1),
    'Age': np.random.randint(25, 75, num_patients),
    'Gender': np.random.choice(['M', 'F'], num_patients),
    'Treatment_Group': np.random.choice(['A', 'B', 'Control'], num_patients)
})

# Generate daily measurements for each patient
measurements = ['BP', 'Glucose', 'Heart_Rate', 'Temperature']
times = ['Morning', 'Afternoon', 'Evening']
measurement_data = {}


# Generate all measurements at once
for day in range(1, days + 1):
    for measurement in measurements:
        for time in times:
            col_name = f'{measurement}_{time}_Day{day}'

            # Generate realistic values based on measurement type
            if measurement == 'BP':
                values = np.random.randint(110, 140, num_patients)
            elif measurement == 'Glucose':
                values = np.random.randint(80, 130, num_patients)
            elif measurement == 'Heart_Rate':
                values = np.random.randint(60, 100, num_patients)
            else:  # Temperature
                values = np.random.normal(37, 0.5, num_patients).round(1)

            measurement_data[col_name] = values

# Create the full DataFrame at once by combining base data and measurements
wide_data = pd.concat([
    patient_base,
    pd.DataFrame(measurement_data)
], axis=1)

print("Wide Format Data Shape:", wide_data.shape)
print("\nFirst few columns of first few rows:")
wide_data.iloc[:5, :10]



Wide Format Data Shape: (500, 364)

First few columns of first few rows:


,Patient_ID,Age,Gender,Treatment_Group,BP_Morning_Day1,BP_Afternoon_Day1,BP_Evening_Day1,Glucose_Morning_Day1,Glucose_Afternoon_Day1,Glucose_Evening_Day1
0,1,63,F,B,118,125,115,99,116,85
1,2,53,M,A,111,115,114,97,103,106
2,3,39,M,Control,130,128,124,108,128,81
3,4,67,F,Control,121,139,131,85,98,108
4,5,32,M,Control,112,116,124,111,104,117


In [ ]:
# Step 1: Identify core columns that should remain as-is
# Hint: These are your patient identifiers and demographics



# Step 2: Plan your melt operation
# Hint: What should be your id_vars?
# Hint: What will your variable name represent?




# Step 3: Clean up the melted data
# Hint: Use str.extract() to split measurement info into components
# Split the Measurement_Info into components





# Step 4: Analysis Preparation:

# Consider how to group the data meaningfully
# Think about which statistical measures would be most informative
# Plan how to handle multiple measurements per patient
# Analysis example: Average measurements by treatment group and time of day
